# BWSN-1 PPO — Colab launcher

Reproducible **launcher** for PPO training of the frozen BWSN-1 environment.
The research logic lives in `experiments/ppo/*` (version-controlled); this
notebook only *calls* it. Run the cells top-to-bottom. See
`experiments/colab/README.md` for the full procedure and scientific boundaries.

**Edit `REPO_URL` and `COMMIT` below to the exact revision validated locally.**


In [ ]:
REPO_URL = 'https://github.com/<you>/<repo>.git'  # TODO
COMMIT   = '<COMMIT>'                               # TODO: exact local commit


## 1. (optional) Mount Google Drive for persistent checkpoints/results


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Point run outputs at Drive so a runtime reset does not lose checkpoints:
# e.g. after cloning:  !ln -s /content/drive/MyDrive/bwsn_runs /content/Thesis/experiments/ppo/runs


## 2. Clone the repo at the validated revision


In [ ]:
%cd /content
!test -d Thesis || git clone $REPO_URL Thesis
%cd /content/Thesis
!git checkout $COMMIT
!git rev-parse --short HEAD


## 3. Install deps ON TOP OF Colab's preinstalled CUDA torch (do NOT reinstall torch)


In [ ]:
!pip -q install -r experiments/colab/requirements-colab.txt
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())


## 4. Verify env + GPU + hydraulic anchor + SB3 contract (authoritative gate)
Expects `cuda_available: true` on a GPU runtime, working-INP SHA match, and the
native leak anchor 84.72988433677985 GPM at rel < 5e-5. **If it fails, STOP.**


In [ ]:
!python -m experiments.ppo.verify_env --full


## 5. (optional) Full env unit battery


In [ ]:
!python -m unittest tests.test_action_mapper tests.test_metrics_reward \
    tests.test_safety tests.test_baselines tests.test_env_integration \
    tests.test_ppo_traceability


## 6. Development training run
DEVELOPMENT evidence (pipeline check), NOT a thesis result until STEP-10 is frozen.


In [ ]:
!python -m experiments.ppo.train --experiment-id dev_run_s0 \
    --total-timesteps 50000 --seed 0 --device auto --checkpoint-freq 5000


## 7. Resume after a runtime reset (checkpoints should be on Drive)


In [ ]:
!python -m experiments.ppo.train --experiment-id dev_run_s0 \
    --resume experiments/ppo/runs/dev_run_s0/checkpoints/ppo_5000_steps.zip \
    --total-timesteps 50000


## 8. Deterministic evaluation (frozen metrics only)


In [ ]:
!python -m experiments.ppo.evaluate \
    --model experiments/ppo/runs/dev_run_s0/final_model.zip \
    --experiment-id dev_run_s0 --seed 0


## 9. TensorBoard (optional)


In [ ]:
%load_ext tensorboard
%tensorboard --logdir experiments/ppo/runs/dev_run_s0/logs


## 10. Persist results to Drive


In [ ]:
!cp -r experiments/ppo/runs/dev_run_s0 /content/drive/MyDrive/bwsn_runs/ 2>/dev/null || echo 'set your Drive path'
